# Run Batch Evaluations on a CSV

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/batch-eval). Run the cells in order in a fresh Python 3.12 kernel.

## Install the SDK

Use the platform SDK on its own. Both the CSV upload and the dataset checks use this package.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "futureagi==0.7.0"], check=True);

## Set your credentials

Set the named environment variables or Colab secrets before running, or enter them at the hidden prompts. Values are not displayed. For self-hosting, set `FI_BASE_URL` to your platform API.

In [ ]:
import os
from getpass import getpass


def require_secret(name):
    if os.environ.get(name):
        return
    value = None
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
    if not value:
        value = getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value


for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    require_secret(name)

## Upload the handoffs

Create a new dataset from three synthetic support cases. Keep the dataset ID to reuse this upload.

In [ ]:
import csv
from uuid import uuid4

from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

name = f"ridgeline-handoffs-{uuid4().hex[:8]}"
path = f"{name}.csv"
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["case", "handoff", "contact"])
    writer.writerows([
        ("return-label", "Return label ready", "returns@example.com"),
        ("order-status", '{"status":"shipped"}', "shipping at example.com"),
        ("repair", '{"status":"received"}', "repairs@example.com"),
    ])

dataset = Dataset(dataset_config=DatasetConfig(
    name=name, model_type=ModelTypes.GENERATIVE_LLM,
))
dataset.create(source=path)
print(dataset.dataset_config.id)

## Map and run both checks

Save the scoring script. It maps `handoff` to the JSON check and `contact` to the email check, waits for both, then exports the three rows.

In [ ]:
%%writefile score_cases.py
import argparse
import csv
import time

from fi.platform import Client

parser = argparse.ArgumentParser()
parser.add_argument("dataset_id")
parser.add_argument("--resume", action="store_true", help="Wait and export without starting a run")
args = parser.parse_args()
dataset_id = args.dataset_id
checks = [
    ("is_json", "handoff", "handoff-json"),
    ("is_email", "contact", "contact-email"),
]


def find_template(client, name):
    page = 0
    while True:
        batch = client.evals.templates.list(
            owner_filter="system", search=name, page=page, page_size=20,
        )
        for item in batch.items:
            if item.name == name:
                return client.evals.templates.get(item.id)
        if not batch.items or (page + 1) * batch.page_size >= batch.total:
            raise LookupError(f"No system template named {name}")
        page += 1


with Client() as client:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        table = client.datasets.get(dataset_id, page_size=3)
        upload = table.dataset_config or {}
        state = upload.get("file_processing_status")
        if state == "completed":
            if upload.get("error_columns"):
                raise RuntimeError(f"CSV has upload errors: {dataset_id}")
            break
        if state == "failed":
            raise RuntimeError(f"CSV upload failed: {dataset_id}")
        time.sleep(2)
    else:
        raise TimeoutError(f"CSV still processing: {dataset_id}")

    def attached():
        items = client.datasets.evals.list(dataset_id, eval_type="user").evals
        return {item["name"]: item for item in items}

    names = [check_name for _, _, check_name in checks]
    columns = {c.name: c.id for c in table.column_config}
    if not args.resume:
        for template_name, column, check_name in checks:
            template = find_template(client, template_name)
            if set(template.required_keys) != {"text"}:
                raise ValueError(f"{template_name} requires {template.required_keys}")
            client.datasets.evals.add(
                dataset_id, name=check_name, template_id=template.id,
                config={"mapping": {"text": columns[column]}}, run=False,
            )

        metrics = attached()
        client.datasets.evals.run(dataset_id, user_eval_ids=[metrics[n]["id"] for n in names])

    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        metrics = attached()
        states = {n: metrics[n]["status"] for n in names}
        if all(s == "Completed" for s in states.values()):
            break
        if any(s not in {"NotStarted", "Queued", "Running", "Completed"} for s in states.values()):
            raise RuntimeError(states)
        time.sleep(2)
    else:
        raise TimeoutError(f"Run again with --resume to check {dataset_id}: {states}")

    table = client.datasets.get(dataset_id, page_size=3)
    columns = {c.name: c.id for c in table.column_config}
    fields = ["case", "handoff", "contact", *names]
    path = f"ridgeline-handoffs-{dataset_id}-scored.csv"
    records = []
    for row in table.table or []:
        cells = row.model_dump()
        record = {}
        for field in fields:
            cell = cells.get(columns[field], {})
            value = cell.get("cell_value")
            if cell.get("status") == "error" or value is None or value == "":
                raise RuntimeError(f"Missing or failed {field} in row {row.row_id}")
            record[field] = value
        records.append(record)
    if len(records) != 3:
        raise RuntimeError(f"Expected 3 scored rows, got {len(records)}")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    print(path)

In [ ]:
dataset_id = str(dataset.dataset_config.id)
scored_path = subprocess.run(
    [sys.executable, "score_cases.py", dataset_id], check=True, stdout=subprocess.PIPE, text=True,
).stdout.strip()
print(scored_path)

## Read the scored CSV

The return-label case should fail only the JSON check, the order-status case only the email check, and the repair case neither.

In [ ]:
with open(f"ridgeline-handoffs-{dataset_id}-scored.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row["case"], row["handoff-json"], row["contact-email"])